# 📂 유저 세션 기반 지원(Apply) 유입 경로 분석

이 노트북은 다음 로직을 수행합니다:
1. **세션 정의**: 유저별로 30분 이상의 유휴 시간이 발생하면 새로운 세션으로 구분합니다.
2. **타겟 필터링**: `POST` 메서드이면서 `api/jobs/id/apply/` 경로인 '진짜 지원 액션'을 추출합니다.
3. **경로 추적**: 각 지원 액션 바로 직전에 발생한 이벤트(Previous Event)를 찾아 유입 경로를 파악합니다.

In [10]:
import polars as pl

# 데이터 로드
df = pl.read_parquet("parquet_logs_cleaned/log_2022-01.parquet")

print(f"📊 원본 데이터 행 수: {len(df):,} 개")

📊 원본 데이터 행 수: 1,023,598 개


In [11]:
# 전처리 블록 (정합성 확인, dt로 변경, dt 기준 정렬)
# 1. 정합성 확인: timestamp 길이가 최소 19자 이상인 정상 데이터만 추리기
df_step1 = df.filter(pl.col("timestamp").str.len_chars() >= 19)

# 2. 19자리까지만 자르기 (예: "2022-01-03 08:57:37")
df_step1 = df_step1.with_columns(
    pl.col("timestamp").str.slice(0, 19).alias("ts_str")
)

# 3. 문자열을 datetime(dt) 타입으로 변환
df_step1 = df_step1.with_columns(
    pl.col("ts_str").str.to_datetime("%Y-%m-%d %H:%M:%S").alias("ts")
)

# 4. 유저 및 시간 기준으로 정렬
df_step1 = df_step1.sort(["user_uuid", "ts"])

# 5. 전처리 결과물 행 수 출력
print(f"📊 전처리 후 정상 데이터 행 수: {len(df_step1):,} 개")
print(f"🗑️ 버려진 이상치 데이터 수: {len(df) - len(df_step1):,} 개")

📊 전처리 후 정상 데이터 행 수: 1,023,559 개
🗑️ 버려진 이상치 데이터 수: 39 개


In [ ]:
# 세션 열(시간 차이) 생성 블록
# 1. 유저별로 '직전 이벤트의 시간'을 옆 칸으로 끌고 오기
df_step2 = df_step1.with_columns(
    pl.col("ts").shift(1).over("user_uuid").alias("prev_ts")
)

# 2. 현재 시간과 직전 시간의 차이를 '분(minutes)' 단위로 계산하기
df_step2 = df_step2.with_columns(
    (pl.col("ts") - pl.col("prev_ts")).dt.total_minutes().alias("diff_min")
)

In [ ]:
# 세션 분리 블록 (30분 유휴시간 기준)

# 1. 시간 차이가 30분 이상이거나, 첫 이벤트(null)이면 새로운 세션(1)으로 표시
df_step3 = df_step2.with_columns(
    ((pl.col("diff_min") > 30) | pl.col("diff_min").is_null()).cast(pl.Int32).alias("is_new_session")
)

# 2. 누적합(cum_sum)을 사용해 세션 ID 번호 매기기 (1, 1, 1, 2, 2, 3...)
df_step3 = df_step3.with_columns(
    pl.col("is_new_session").cum_sum().over("user_uuid").alias("session_id")
)

In [19]:
# 3. 세션 예시 출력
sample_user = '1248b20b-e9e8-4e06-8c0d-c273b709d082'  # 맨 첫 번째 유저 선택
sample_df = df_step3.filter(pl.col("user_uuid") == sample_user)

print(f"👀 샘플 유저 ({sample_user}) 의 행동 및 세션 분리 결과:")
with pl.Config(tbl_rows=10000, fmt_str_lengths=60):
    # 필요한 컬럼만 깔끔하게 추려서 보기
    print(sample_df.select(["user_uuid", "URL", "method", "ts", "diff_min", "session_id"]))

👀 샘플 유저 (1248b20b-e9e8-4e06-8c0d-c273b709d082) 의 행동 및 세션 분리 결과:
shape: (9_007, 6)
┌─────────────────────┬─────────────────────┬────────┬─────────────────────┬──────────┬────────────┐
│ user_uuid           ┆ URL                 ┆ method ┆ ts                  ┆ diff_min ┆ session_id │
│ ---                 ┆ ---                 ┆ ---    ┆ ---                 ┆ ---      ┆ ---        │
│ str                 ┆ str                 ┆ str    ┆ datetime[μs]        ┆ i64      ┆ i32        │
╞═════════════════════╪═════════════════════╪════════╪═════════════════════╪══════════╪════════════╡
│ 1248b20b-e9e8-4e06- ┆ admin               ┆ GET    ┆ 2022-01-03 01:11:15 ┆ null     ┆ 1          │
│ 8c0d-c273b709d082   ┆                     ┆        ┆                     ┆          ┆            │
│ 1248b20b-e9e8-4e06- ┆ admin/jsi18n        ┆ GET    ┆ 2022-01-03 01:11:22 ┆ 0        ┆ 1          │
│ 8c0d-c273b709d082   ┆                     ┆        ┆                     ┆          ┆            │
│ 1248b20

In [29]:
# 첫 이벤트부터 Apply 1단계(POST) 까지의 이벤트 집계 블록

# 1. '지원 1단계 완료(POST)' 조건 정의
is_apply1 = (pl.col("URL") == "api/jobs/id/apply/step1") & (pl.col("method") == "POST")

# 2. 각 세션별로 '지원 1단계'가 발생한 가장 첫 번째 시간(min)을 찾아서 기록
df_step4 = df_step3.with_columns(
    pl.col("ts").filter(is_apply1).min().over(["user_uuid", "session_id"]).alias("apply1_time")
)

# 3. 지원 1단계를 수행한 적이 있는 세션만 남기기
df_step4 = df_step4.filter(pl.col("apply1_time").is_not_null())

# 4. [핵심] 해당 세션 안에서 '지원 1단계 시간'보다 같거나 이전인 이벤트만 싹 추려내기!
df_step4 = df_step4.filter(pl.col("ts") <= pl.col("apply1_time"))
df_step4 = df_step4.sort(["user_uuid", "session_id", "ts"])

print(f"📊 Apply 1단계(POST) 이전으로 추려진 최종 이벤트 수: {len(df_step4):,} 개")

export_filename = "filtered_apply_sessions.parquet"
df_step4.write_parquet(export_filename)

print(f"💾 필터링된 세션 데이터가 성공적으로 저장되었습니다: {export_filename}")

📊 Apply 1단계(POST) 이전으로 추려진 최종 이벤트 수: 111,081 개
💾 필터링된 세션 데이터가 성공적으로 저장되었습니다: filtered_apply_sessions.parquet


In [31]:
import polars as pl

# 1. 먼저 데이터를 유저, 세션, 시간(ts) 순으로 철저히 정렬합니다.
df = df.sort(["user_uuid", "session_id", "ts"])

# 2. 우리가 찾고자 하는 타겟 이벤트를 정의합니다.
is_apply1 = (pl.col("URL") == "api/jobs/id/apply/step1") & (pl.col("method") == "POST")

# 3. 핵심 로직: 누적합(cum_sum)을 이용해 타겟 이벤트 이후를 날려버립니다.
df = (
    df.with_columns(
        # 타겟 이벤트면 1, 아니면 0을 반환하고 누적합을 구합니다.
        # 이렇게 하면 타겟 이전은 0, 첫 타겟은 1, 그 이후의 행들도 1(또는 그 이상)이 됩니다.
        pl.when(is_apply1).then(1).otherwise(0)
          .cum_sum()
          .over(["user_uuid", "session_id"])
          .alias("target_seen")
    )
    # 세션 내에 타겟 이벤트가 아예 없는 경우(전환 실패 유저) 통째로 버림
    .filter(pl.col("target_seen").max().over(["user_uuid", "session_id"]) > 0)
    
    # 누적합이 0인 구간(타겟 이벤트 이전)과, 
    # 누적합이 1이면서 정확히 타겟 이벤트인 그 "한 줄"까지만 살립니다!
    # 동일한 타임스탬프를 가진 step2가 그 뒤에 오더라도 is_apply1이 False이므로 무조건 잘려나갑니다.
    .filter(
        (pl.col("target_seen") == 0) | 
        ((pl.col("target_seen") == 1) & is_apply1)
    )
)

# 4. 이제 정렬 순서를 유지하며 경로를 묶습니다.
path_agg = df.group_by(["user_uuid", "session_id"], maintain_order=True).agg(
    pl.col("URL").alias("path_list")
)

# 5. 리스트를 문자열로 결합
path_agg = path_agg.with_columns(
    pl.col("path_list").list.join(" -> ").alias("full_path")
)

# 6. 결과 집계
final_result = path_agg.group_by("full_path").len(name="count").sort("count", descending=True)

# 출력
with pl.Config(tbl_rows=50, fmt_str_lengths=150):
    print(final_result.head(20))

shape: (20, 2)
┌──────────────────────────────────────────────────────────────────────────────────────────┬───────┐
│ full_path                                                                                ┆ count │
│ ---                                                                                      ┆ ---   │
│ str                                                                                      ┆ u32   │
╞══════════════════════════════════════════════════════════════════════════════════════════╪═══════╡
│ api/jobs/id/apply/step1                                                                  ┆ 15    │
│ jobs/id/apply/step1 -> api/recommend_specialty -> api/jobs/id/apply/step1                ┆ 11    │
│ jobs/id/apply/step1 -> api/recommend_specialty -> jobs/id/apply/step2 ->                 ┆ 7     │
│ api/jobs/id/apply/step1                                                                  ┆       │
│ api/recommend_specialty -> api/jobs/id/apply/step1                        

In [25]:
import polars as pl
import glob
import time
import os

print("=====================================================================")
print("🚀 [안전 모드] 24개 파일 개별 처리 및 경로 집계 시작...")
print("=====================================================================")

start_time = time.time()

# 1. 처리할 모든 파케이 파일 목록 불러오기
file_list = sorted(glob.glob("parquet_logs_cleaned/log_*.parquet"))

# 2. 각 파일의 '요약 결과(경로별 카운트)'만 담을 빈 리스트 생성
all_monthly_results = []

# 3. for문을 돌면서 하나씩 차근차근 처리
for file in file_list:
    filename = os.path.basename(file)
    
    # 데이터 로드
    df = pl.read_parquet(file)
    
    # [전처리 및 세션 분리]
    df = (
        df.filter(pl.col("timestamp").str.len_chars() >= 19)
        .with_columns(
            pl.col("timestamp").str.slice(0, 19).str.to_datetime("%Y-%m-%d %H:%M:%S").alias("ts")
        )
        .sort(["user_uuid", "ts"])
        .with_columns(
            pl.col("ts").shift(1).over("user_uuid").alias("prev_ts")
        )
        .with_columns(
            (pl.col("ts") - pl.col("prev_ts")).dt.total_minutes().alias("diff_min")
        )
        .with_columns(
            ((pl.col("diff_min") > 30) | pl.col("diff_min").is_null()).cast(pl.Int32).alias("is_new_session")
        )
        .with_columns(
            pl.col("is_new_session").cum_sum().over("user_uuid").alias("session_id")
        )
    )
    
    # [지원 1단계(POST) 직전까지의 경로 추려내기]
    is_apply1 = (pl.col("URL") == "api/jobs/id/apply/step1") & (pl.col("method") == "POST")
    
    df = (
        df.with_columns(
            pl.col("ts").filter(is_apply1).min().over(["user_uuid", "session_id"]).alias("apply1_time")
        )
        .filter(pl.col("apply1_time").is_not_null())
        .filter(pl.col("ts") <= pl.col("apply1_time"))
    )
    
    # [해당 월(파일)의 경로별 빈도수 집계]
    # 전체 로그가 아닌 "어떤 경로가 몇 번 나왔다"는 수천 줄짜리 요약본만 만듭니다.
    monthly_path_counts = (
        df.group_by(["user_uuid", "session_id"])
        .agg(pl.col("URL").alias("path_list"))
        .with_columns(pl.col("path_list").list.join(" -> ").alias("full_path"))
        .group_by("full_path")
        .len(name="count")
    )
    
    # 4. 요약된 결과만 리스트에 추가 (메모리 극대화)
    all_monthly_results.append(monthly_path_counts)
    print(f"✔️ {filename} 처리 완료 (추출된 고유 경로 수: {len(monthly_path_counts):,}개)")

# =====================================================================
# [최종 취합 단계]
# =====================================================================
print("\n🔄 모든 파일 처리가 완료되었습니다. 전체 결과를 하나로 병합합니다...")

# 5. 리스트에 담긴 24개의 데이터프레임을 하나로 합치고, full_path 기준으로 다시 한번 총합(sum) 계산
final_result = (
    pl.concat(all_monthly_results)
    .group_by("full_path")
    .sum()  # 각 파일에서 계산된 count들을 하나로 더함
    .sort("count", descending=True)
)

end_time = time.time()
print(f"✅ 전체 분석 완료! (총 소요 시간: {end_time - start_time:.2f}초)\n")

# 결과 출력
print("🎯 [전체 기간] 처음 유입 ~ 지원 1단계(POST) 완료 경로 TOP 20")
with pl.Config(tbl_rows=20, fmt_str_lengths=150):
    print(final_result.head(20))

# 저장
final_result.write_csv("total_apply_funnel_paths.csv")
print("💾 'total_apply_funnel_paths.csv' 파일이 저장되었습니다!")

🚀 [안전 모드] 24개 파일 개별 처리 및 경로 집계 시작...
✔️ log_2022-01.parquet 처리 완료 (추출된 고유 경로 수: 2,668개)
✔️ log_2022-02.parquet 처리 완료 (추출된 고유 경로 수: 2,454개)
✔️ log_2022-03.parquet 처리 완료 (추출된 고유 경로 수: 2,757개)
✔️ log_2022-04.parquet 처리 완료 (추출된 고유 경로 수: 2,684개)
✔️ log_2022-05.parquet 처리 완료 (추출된 고유 경로 수: 2,628개)
✔️ log_2022-06.parquet 처리 완료 (추출된 고유 경로 수: 2,274개)
✔️ log_2022-07.parquet 처리 완료 (추출된 고유 경로 수: 2,148개)
✔️ log_2022-08.parquet 처리 완료 (추출된 고유 경로 수: 2,240개)
✔️ log_2022-09.parquet 처리 완료 (추출된 고유 경로 수: 1,935개)
✔️ log_2022-10.parquet 처리 완료 (추출된 고유 경로 수: 2,045개)
✔️ log_2022-11.parquet 처리 완료 (추출된 고유 경로 수: 1,989개)
✔️ log_2022-12.parquet 처리 완료 (추출된 고유 경로 수: 2,683개)
✔️ log_2023-01.parquet 처리 완료 (추출된 고유 경로 수: 2,192개)
✔️ log_2023-02.parquet 처리 완료 (추출된 고유 경로 수: 3,440개)
✔️ log_2023-03.parquet 처리 완료 (추출된 고유 경로 수: 3,571개)
✔️ log_2023-04.parquet 처리 완료 (추출된 고유 경로 수: 3,361개)
✔️ log_2023-05.parquet 처리 완료 (추출된 고유 경로 수: 3,094개)
✔️ log_2023-06.parquet 처리 완료 (추출된 고유 경로 수: 2,181개)
✔️ log_2023-07.parquet 처리 완료 (추출된 고유 경로 수: 2,